# HIPAA Compliance Knowledge Assistant for HR
### Using Semantic Search and Retrieval-Augmented Generation (RAG) to Support Workforce Compliance

**Author:** Veronica Jenkins  |  BANA 6370 Programming with AI, University of Dallas  |  Fall 2026
**Data:** HIPAA Compliance Training Dataset (Troy, 2025), Hugging Face, CC0 public domain
**Tools:** Python, Hugging Face, IBM Granite (embedding, reranker, and language models), FAISS

---

## Executive Summary

**The business problem.** In a healthcare organization, HR owns much of the workforce side of HIPAA: new-hire and annual privacy training, sanction policies for employees who violate privacy rules, and the day-to-day questions managers and employees bring to HR. The answers live in thousands of pages of federal regulations and guidance. HR generalists rarely have time to search them, and a wrong answer can lead to inconsistent discipline, audit findings, or legal exposure.

**What I built.** A prototype knowledge assistant that takes a plain-English HR question, finds the most relevant passages across eight federal HIPAA sources, and drafts an answer grounded in those passages with citations back to the source regulation.

**Key findings.**
1. **Search model choice mattered more than anything else.** On a test built from the dataset's own questions, the default model from the course example found almost none of the correct passages. An enterprise retrieval model (IBM Granite Embedding R2) performed far better, and adding a second-stage reranker roughly tripled its ranking score (MRR, 0.100 → 0.301).
2. **A modern language model produced executive-ready answers.** It followed a structured Summary / Requirements / Recommended Actions format that the course's original model could not.
3. **The same model made confident, convincing errors.** It invented examples of discipline that the regulations do not list, cited the wrong CFR section, and misclassified security safeguards.

**Recommendation.** Pilot the assistant as a *research aid* for HR and Compliance staff, not as a source of final answers. Every answer used for policy or discipline decisions should be reviewed by the Privacy or Compliance Officer, with citations checked against the official regulation text.

---

## Business Context

| Stakeholder | What they need from this tool |
|---|---|
| CHRO / HR leadership | Confidence that workforce HIPAA programs meet requirements; lower compliance risk |
| HR business partners and generalists | Fast, accurate answers to manager and employee questions |
| Privacy / Compliance Officer | Traceable answers they can verify; fewer routine questions |
| People managers | Clear guidance when an employee mishandles patient information |

**Example questions the assistant is designed to answer:** What HIPAA training must our workforce receive? What sanctions are required when an employee violates privacy policy? What must we do when a breach is discovered?

**Scope note.** HIPAA's definition of protected health information excludes employment records a covered entity holds in its role as an employer. This assistant supports HR's role in *workforce compliance* (training, sanctions, incident response), not the management of personnel files.

**Limitations of the data.** The question-and-answer pairs were generated from official federal publications, but the questions themselves were machine-written. This is a prototype for learning and demonstration, and its outputs are not legal advice.

## 1. Setup and Environment

**Business question:** What tools does the assistant need, and how do we make sure the analysis can be repeated and audited?

**What we did:**
- Loaded the standard open-source toolkit for this kind of work: `datasets` (data loading), `sentence_transformers` (search models), `transformers` and `torch` (language model), `faiss` (fast search index), and `pandas` / `numpy` (data handling).
- Connected Google Drive as a **save point**, so completed work survives if the cloud session disconnects.
- Set the random seed to 41 for Python, NumPy, and PyTorch. The seed makes random steps repeatable; it does not shuffle or change the data. In this notebook the only random step is drawing the test questions in the bonus evaluation, and because all 56 eligible questions are used, the seed affects only their order, not which questions are tested.
- Answer generation is repeatable for a different reason: the language model uses **greedy decoding** (`do_sample=False`), which always picks the most likely next word. That, not the seed, is why the same question produces the same answer on every run.
- Checked whether a GPU (a graphics processor that speeds up AI models) is available.

**Why it matters for HR:** Compliance work has to be repeatable. If an auditor or Compliance Officer asks how an answer was produced, we need to be able to re-run the same analysis and get the same result.

**How to read the results:** `Using device: cuda` means the GPU is active. `cpu` means the notebook will still run, just more slowly.

In [1]:
# Install/upgrade packages BEFORE importing them
!pip install -q -U transformers sentence-transformers
!pip install -q faiss-cpu

import os
import re
import random
import textwrap
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

import faiss
from typing import List, Tuple

# --- Google Drive checkpoint folder (same approach as Unit 3) ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = "/content/drive/MyDrive/BANA6370_Lab4"
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_FOLDER}")

# --- Reproducibility: seed 41 ---
RANDOM_SEED = 41
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# --- Keep long text readable on screen AND in the exported PDF ---
pd.set_option("display.max_colwidth", 80)
def wrap(text: str, width: int = 88) -> str:
    """Wrap long text so it doesn't run off the page in the PDF export."""
    return "\n".join(textwrap.fill(line, width) for line in str(text).splitlines())


# Optional: check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 20.0 MB/s eta 0:00:00
ERROR: Operation cancelled by user
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.3 MB/s eta 0:00:00
Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/BANA6370_Lab4
Using device: cuda


## 2. Data Loading and Inspection

**Business question:** What knowledge will the assistant draw from, and is it the right material for HR's questions?

**What we did:** Loaded 1,287 question-and-answer records built from nine federal publications: the HIPAA Privacy Rule, Security Rule, Breach Notification Rule, HITECH Omnibus Rule, NIST implementation guidance, and FDA medical device guidance. For each record we kept:
- **The answer text**, which becomes a "knowledge article" in the assistant's library
- **The original question**, which we use later to test search accuracy
- **Metadata**: source document, question type, and CFR section when available

**Why it matters for HR:** The assistant can only be as good as its library. Knowing what is (and isn't) in it tells us which questions it can answer well.

**How to read the results:** The source counts show that the **Privacy Rule makes up roughly 44% of the library**, while the Security Rule, which covers most workforce training and sanction requirements, has far fewer records. That imbalance matters. Security questions may return Privacy Rule passages simply because there are more of them. The sample records confirm the text is real regulatory language.

In [2]:
# Load the HIPAA Compliance dataset (combine train + validation into one knowledge base)
raw = load_dataset("ethanolivertroy/hipaa-compliance-training")
print(raw)

df = pd.concat(
    [raw[split].to_pandas().assign(split=split) for split in raw.keys()],
    ignore_index=True
)
print(f"\nNumber of documents (train + validation combined): {len(df)}")

# Each row stores a chat: system prompt, user question, assistant answer.
# The system prompt is identical on every row, so we keep only the question and answer.
def get_turn(messages, role: str) -> str:
    for m in messages:
        if m["role"] == role:
            return m["content"]
    return ""

df["question"] = df["messages"].apply(lambda m: get_turn(m, "user"))
df["text"]     = df["messages"].apply(lambda m: get_turn(m, "assistant"))  # the searchable knowledge

# Pull useful metadata out of the nested 'metadata' dictionary
def meta(m, key):
    return m.get(key) if isinstance(m, dict) else None

df["source"]      = df["metadata"].apply(lambda m: meta(m, "source"))
df["qtype"]       = df["metadata"].apply(lambda m: meta(m, "type"))
df["cfr_section"] = df["metadata"].apply(lambda m: meta(m, "cfr_section"))

# Map source codes to human-readable document names
source_names = {
    "privacy_rule_2000":        "Privacy Rule (2000)",
    "privacy_mod_2002":         "Privacy Rule Modifications (2002)",
    "security_rule_2003":       "Security Rule (2003)",
    "breach_notification_2009": "Breach Notification Rule (2009)",
    "hitech_omnibus_2013":      "HITECH Omnibus Rule (2013)",
    "nist_sp800-66r2":          "NIST SP 800-66 Rev. 2 (2024)",
    "fda_postmarket_2016":      "FDA Postmarket Cybersecurity (2016)",
    "fda_premarket_2023":       "FDA Premarket Cybersecurity (2023)",
    "45cfr160":                 "45 CFR Part 160",
}
df["category"] = df["source"].map(source_names).fillna(df["source"])

df = df[["question", "text", "category", "source", "qtype", "cfr_section", "split"]]

print("\nDocuments by source:")
print(df["category"].value_counts().to_string())

# Show a few example documents
print("\nSample documents:\n")
for i in range(3):
    print(f"ID: {i}")
    print(f"Source: {df.loc[i, 'category']}  |  Question type: {df.loc[i, 'qtype']}")
    print(f"Question: {wrap(df.loc[i, 'question'])}")
    print(f"Content: {wrap(df.loc[i, 'text'][:600])}...\n")
    print("-" * 80)

README.md:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/2.62M [00:00<?, ?B/s]

valid.jsonl:   0%|          | 0.00/656k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1029 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/258 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 1029
    })
    validation: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 258
    })
})

Number of documents (train + validation combined): 1287

Documents by source:
category
Privacy Rule (2000)                    567
HITECH Omnibus Rule (2013)             229
Privacy Rule Modifications (2002)      151
NIST SP 800-66 Rev. 2 (2024)           102
Security Rule (2003)                    85
Breach Notification Rule (2009)         65
FDA Premarket Cybersecurity (2023)      51
FDA Postmarket Cybersecurity (2016)     28
45 CFR Part 160                          3
45cfr164                                 3
45cfr162                                 3

Sample documents:

ID: 0
Source: NIST SP 800-66 Rev. 2 (2024)  |  Question type: semantic_chunk
Question: What does nist_sp800-66r2 say about HIPAA compliance? (Part 25)
Content: According to nist_sp800-66r2:

Identify All ePHI

## 3. Data Cleaning

**Business question:** Is the library clean enough to trust?

**What we did:** Applied light, documented cleaning steps. Nothing was rewritten, so the original regulatory wording is preserved.

| Step | Decision | Reason |
|---|---|---|
| 1 | Collapse extra spaces and line breaks | Consistent formatting for the search model |
| 2 | Remove empty records | No content to search |
| 3 | Remove 45 CFR Parts 160, 162, 164 records | The dataset's documentation reports these files failed to extract. The printed example confirms it: it is website menu text, not regulation |
| 4 | Remove records under 80 characters | Too short to answer a question |
| 5 | Remove exact duplicate answers | Duplicates crowd out other relevant results |

**A decision to note:** Some records use generic questions like *"What does [document] say about HIPAA compliance?"* We **kept** their answer text in the library, because it is real regulatory content. We **excluded** them from the accuracy test in Section 6, because a generic question has no single correct answer.

**Why it matters for HR:** Clean inputs mean fewer irrelevant or repeated results for the person asking the question.

**How to read the results:** The cleaning table shows how many records remained after each step. Most removals come from duplicate answers.

In [3]:
counts = {"Starting documents": len(df)}

# 1. Basic normalization: collapse extra spaces/line breaks
df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["question"] = df["question"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# 2. Remove empty documents
df = df[df["text"] != ""].reset_index(drop=True)
counts["After removing empty documents"] = len(df)

# 3. Remove the sources the dataset card reports as failed extractions (45 CFR Parts 160, 162, 164)
bad_source = df[df["source"].astype(str).str.startswith("45cfr")]
if len(bad_source) > 0:
    print(f"Found {len(bad_source)} documents from failed-extraction sources. Example (inspect before dropping):")
    print(wrap(bad_source.iloc[0]["text"][:400]), "\n")
df = df[~df["source"].astype(str).str.startswith("45cfr")].reset_index(drop=True)
counts["After removing failed-extraction sources (45 CFR 160/162/164)"] = len(df)

# 4. Remove very short documents that carry too little information to be useful
MIN_CHARS = 80
df = df[df["text"].str.len() >= MIN_CHARS].reset_index(drop=True)
counts[f"After removing very short documents (< {MIN_CHARS} chars)"] = len(df)

# 5. Remove exact duplicate answers so search results aren't repetitive
df = df.drop_duplicates(subset="text").reset_index(drop=True)
counts["After removing duplicate documents"] = len(df)

display(pd.DataFrame(list(counts.items()), columns=["Cleaning step", "Documents remaining"]))

before_count, after_count = counts["Starting documents"], len(df)
print(f"\nDocuments before cleaning: {before_count}")
print(f"Documents after cleaning:  {after_count}")

print("\nExample cleaned document:\n")
print(wrap(df.loc[0, "text"][:800]))

Found 9 documents from failed-extraction sources. Example (inspect before dropping):
According to 45cfr160: [Skip to main content](#main-content) Menu [![GovInfo
logo](/media/GovInfo_logo_gray.png)](/) [Search](#searchHolder) * Browse + [A to
Z](/browse-a-z) + [Category](/app/browse/category) + [Date](/app/browse/date) +
[Committee](/browse/committee) + [Author](/app/browse/author) * About + [About](/about)
+ [Policies](/about/policies) + [Authentication](/about/authentication) + 



,Cleaning step,Documents remaining
0,Starting documents,1287
1,After removing empty documents,1287
2,After removing failed-extraction sources (45 CFR 160/162/164),1278
3,After removing very short documents (< 80 chars),1277
4,After removing duplicate documents,1211



Documents before cleaning: 1287
Documents after cleaning:  1211

Example cleaned document:

According to nist_sp800-66r2: Identify All ePHI and Relevant Information Systems | •
Identify where ePHI is generated within the organization, where it enters the
organization, where it moves within the organization, where it is stored, and where it
leaves the organization. • Identify all systems 30 that house ePHI. Be sure to identify
mobile devices, medical equipment, and IoT devices that store, process, or transmit
ePHI. • Include all hardware and software that are used to collect, store, process, or
transmit ePHI. • Analyze business functions and verify the ownership and control of
information system elements as necessary. • Consider the impact of a merger or
acquisition on risks to ePHI. During a merger or acquisition, new data pathways may be
introduced that lead to ePHI being stored


## 4. Document Embeddings

**Business question:** How does a computer find passages by *meaning* rather than exact keywords?

**What we did:** Converted each passage into an **embedding**, a list of 768 numbers that works like a "meaning fingerprint." Passages about similar topics get similar fingerprints, even when they use different words. For example, "sanctions for workforce members" and "disciplining employees who violate policy" land close together.

**Model choice:** We replaced the course example's model (MiniLM) with **IBM Granite Embedding English R2**:
- **It reads the whole passage.** MiniLM stops reading at about 256 tokens (roughly 190 words), and many of these passages are longer. Granite reads up to 8,192 tokens.
- **License-clean training data.** IBM trained it only on data with commercial-friendly licenses, which matters when a tool will be used inside a regulated organization.

**No model was trained in this notebook.** Granite Embedding R2 is *pretrained*: IBM already trained it on large text collections, and we use it as-is to convert our passages into embeddings. The same is true of the reranker and the language model. Nothing here trains or fine-tunes a model, so any weak results reflect how well a pretrained model fits *this* data, not a failed training step.

The results are saved to Google Drive, so this step does not need to be repeated after a disconnect.

**How to read the results:** The shape `(number of passages, 768)` confirms every passage received a 768-number fingerprint.

In [4]:
# Use a model better suited for HIPAA compliance data
embedding_model_name = "ibm-granite/granite-embedding-english-r2"
model_tag = embedding_model_name.split("/")[-1]
EMB_PATH = f"{DRIVE_FOLDER}/emb_{model_tag}.npy"
DF_PATH  = f"{DRIVE_FOLDER}/hipaa_clean_docs.pkl"
FORCE_REBUILD = True    # set True once if you change the cleaning steps in Cell 3

embedding_model = SentenceTransformer(embedding_model_name, device=device)
print(f"Embedding model max input length: {embedding_model.max_seq_length} tokens")

if os.path.exists(EMB_PATH) and os.path.exists(DF_PATH) and not FORCE_REBUILD:
    embeddings = np.load(EMB_PATH)
    df = pd.read_pickle(DF_PATH)
    print(f"Loaded saved checkpoint from Drive: {EMB_PATH}")
else:
    documents: List[str] = df["text"].tolist()
    print("Generating embeddings...")
    embeddings = embedding_model.encode(
        documents,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True   # unit-length vectors, so L2 distance ranks the same as cosine similarity
    )
    np.save(EMB_PATH, embeddings)
    df.to_pickle(DF_PATH)
    print(f"Checkpoint saved to Drive: {DRIVE_FOLDER}")

print("Embeddings shape:", embeddings.shape)


modules.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  298MB            

...

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model max input length: 8192 tokens
Generating embeddings...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Checkpoint saved to Drive: /content/drive/MyDrive/BANA6370_Lab4
Embeddings shape: (1211, 768)


## 5. Building the Search Index

**Business question:** How do we search the whole library instantly?

**What we did:** Loaded every fingerprint into a **FAISS index**, a search structure built for fast similarity matching. Think of it as the card catalog for the library.

**Why it matters for HR:** Employees and managers expect answers in seconds. This index compares a question against every passage almost instantly, and it would scale to a much larger policy library.

**How to read the results:** The number of vectors in the index should match the number of cleaned passages. That confirms nothing was lost.

In [5]:
# Build FAISS index
embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings.astype("float32"))

print(f"FAISS index built with {index.ntotal} vectors.")


FAISS index built with 1211 vectors.


### How the Search Function Works

When someone asks a question, `semantic_search()`:
1. **Creates a fingerprint of the question**, the same way we fingerprinted the passages.
2. **Finds the closest passages** in the index.
3. **Returns them** with their source document, CFR section, and a **distance score**.

**How to read the distance score:** Lower means a closer match. Scores run from 0 (identical meaning) to 4 (opposite meaning), and in practice good matches fall well below 0.5.

**Demo:** *"What HIPAA training is required for new employees?"* The top results come from NIST's security training guidance and the Security Rule, which is exactly where training requirements live.

In [6]:
def semantic_search(query: str, k: int = 5) -> pd.DataFrame:
    """
    Given a user query, return the top-k most similar documents.
    """
    query_embedding = embedding_model.encode([query],convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(query_embedding.astype("float32"), k)
    indices = indices[0]
    distances = distances[0]

    results = df.iloc[indices].copy()
    results["distance"] = distances
    return results[["category", "cfr_section", "text", "distance"]]

# Demonstration query
demo_query = "What HIPAA training is required for new employees?"
demo_results = semantic_search(demo_query, k=3)
print("Demo query:", demo_query)
demo_results

Demo query: What HIPAA training is required for new employees?


,category,cfr_section,text,distance
573,NIST SP 800-66 Rev. 2 (2024),None,According to nist_sp800-66r2: s made available and are workforce members app...,0.236320
930,Security Rule (2003),None,"According to security_rule_2003: nd procedures, whether through contract lan...",0.251979
1145,NIST SP 800-66 Rev. 2 (2024),None,According to nist_sp800-66r2: Conduct a Training Needs Assessment | • Determ...,0.272190


## 6. Retrieval Evaluation

**Business question:** Before we trust the assistant's answers, does its search actually find the right regulations?

**What we did:** Tested three questions HR teams commonly face:
1. Security awareness training requirements for the workforce
2. Sanctions for employees who violate privacy policies
3. Notifying individuals after a breach of patient information

For each, we pulled the top 3 passages and summarized them in a table.

**Why it matters for HR:** A generated answer can only be as accurate as the passages behind it. If search pulls the wrong regulation, the answer will be wrong no matter how well it is written.

**A caution on evidence:** These three queries were hand-picked, so they *illustrate* how search behaves rather than prove how well it performs. The bonus test after the reranking section measures performance systematically across 56 questions.

**How to read the results:** For each query, check whether the source documents make sense (for example, breach questions should return the Breach Notification Rule) and whether the snippets address the question.

In [7]:
evaluation_queries = [
    "security awareness training requirements for the workforce",
    "sanctions for employees who violate privacy policies",
    "notifying individuals after a breach of patient information",
]

all_eval_results = []
eval_rows = []

for q in evaluation_queries:
    print(f"\n=== Query: {q} ===")
    res = semantic_search(q, k=3)
    for rank, (i, row) in enumerate(res.iterrows(), start=1):
        snippet = row["text"][:200].replace("\n", " ")
        print(f"- Source: {row['category']}, Distance: {row['distance']:.4f}")
        print(wrap(f"  Snippet: {snippet}..."))
        eval_rows.append({
            "Query": q,
            "Rank": rank,
            "Source": row["category"],
            "Distance": round(float(row["distance"]), 4),
            "Snippet": row["text"][:150] + "..."
        })
    all_eval_results.append((q, res))

# Summary table for decision-makers
eval_table = pd.DataFrame(eval_rows)
display(eval_table)



=== Query: security awareness training requirements for the workforce ===
- Source: Security Rule (2003), Distance: 0.1756
  Snippet: According to security_rule_2003: nd procedures, whether through contract
language or other means. Covered entities are not required to provide training to
business associates or anyone else that is no...
- Source: NIST SP 800-66 Rev. 2 (2024), Distance: 0.1873
  Snippet: According to nist_sp800-66r2: Conduct a Training Needs Assessment | •
Determine the training needs of the organization. • Interview and involve key personnel
in assessing security training needs. • Us...
- Source: NIST SP 800-66 Rev. 2 (2024), Distance: 0.2190
  Snippet: According to nist_sp800-66r2: If not, who will deliver the training? • What
is the security training budget? | | 5. Implement the Training | • Schedule and conduct
the training outlined in the strateg...

=== Query: sanctions for employees who violate privacy policies ===
- Source: Privacy Rule (2000), Distance: 0.2285


,Query,Rank,Source,Distance,Snippet
0,security awareness training requirements for the workforce,1,Security Rule (2003),0.1756,"According to security_rule_2003: nd procedures, whether through contract lan..."
1,security awareness training requirements for the workforce,2,NIST SP 800-66 Rev. 2 (2024),0.1873,According to nist_sp800-66r2: Conduct a Training Needs Assessment | • Determ...
2,security awareness training requirements for the workforce,3,NIST SP 800-66 Rev. 2 (2024),0.2190,"According to nist_sp800-66r2: If not, who will deliver the training? • What ..."
3,sanctions for employees who violate privacy policies,1,Privacy Rule (2000),0.2285,According to privacy_rule_2000: e sanctions. These sanctions do not apply to...
4,sanctions for employees who violate privacy policies,2,Security Rule (2003),0.2523,According to security_rule_2003: §164.306(a). - (C) Sanction policy (Require...
5,sanctions for employees who violate privacy policies,3,NIST SP 800-66 Rev. 2 (2024),0.2569,According to nist_sp800-66r2: • Establish a frequency for reviewing policy a...
6,notifying individuals after a breach of patient information,1,Breach Notification Rule (2009),0.1762,According to breach_notification_2009: to notify an affected individual or t...
7,notifying individuals after a breach of patient information,2,Breach Notification Rule (2009),0.1904,According to breach_notification_2009: ragraph (a) of this section without u...
8,notifying individuals after a breach of patient information,3,Breach Notification Rule (2009),0.2086,According to breach_notification_2009: quired by paragraph (a) of this secti...


### Interpretation of Retrieval Quality

- **Breach notification: strongest result.** All three passages came from the Breach Notification Rule, including the requirement to notify individuals no later than 60 calendar days after discovery. An HR or Compliance lead could act on these directly.
- **Sanctions: strong.** Search found the Security Rule's *"Sanction policy (Required)"* standard and the Privacy Rule's exception protecting whistleblowers from sanctions. Together these are the two facts HR most needs when disciplining an employee.
- **Training: mixed.** The top result was related but indirect: it explains that covered entities are *not* required to train business associates. The next two results (NIST's training needs assessment and implementation steps) were directly useful.

**Takeaway:** Search is reliable for well-defined compliance topics. Broader questions can return passages that are related but not the best answer. The next step tests whether a second review stage can fix that.

### Improving Search with a Second Review Stage (Reranking)

**Business question:** Can a more careful second look put the best passage at the top?

**What we did:** Built a **two-stage search**:
1. Fast search pulls the 20 closest passages.
2. A **reranker** (IBM Granite Reranker R2) reads the question and each passage *together* and re-scores how well they match.

**An HR analogy:** This is how recruiting works. An applicant tracking system screens hundreds of résumés by keyword match, then a recruiter reads the top 20 carefully and re-ranks them. The first pass is fast, and the second is more accurate.

**How to read the results:** The rerank score runs from 0 to 1, where higher means a better match. Compare the "before" and "after" tables for the sanctions question.

**What we observed:** The reranker moved a Privacy Rule passage about documenting sanctions and complaints to the top. It also pushed the Security Rule's *"Sanction policy (Required)"* passage out of the top 3, even though that is arguably the single most relevant passage. **A single example can mislead**, so the next cell measures performance across many questions.

In [8]:
# Two-stage retrieval: fast embedding search pulls 20 candidates,
# then a cross-encoder reranker re-reads each (question, passage) pair and re-orders them.
reranker = CrossEncoder("ibm-granite/granite-embedding-reranker-english-r2", device=device)

def reranked_search(query: str, k: int = 3, n_candidates: int = 20) -> pd.DataFrame:
    candidates = semantic_search(query, k=n_candidates)
    candidates["rerank_score"] = reranker.predict([(query, t) for t in candidates["text"]])
    return candidates.sort_values("rerank_score", ascending=False).head(k)

# Before vs. after reranking for one query
demo_q = evaluation_queries[1]
print("Query:", demo_q)
print("\nEmbedding search only (top 3):")
display(semantic_search(demo_q, k=3)[["category", "distance", "text"]])
print("\nAfter reranking (top 3):")
display(reranked_search(demo_q, k=3)[["category", "rerank_score", "text"]])

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  598MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Query: sanctions for employees who violate privacy policies

Embedding search only (top 3):


,category,distance,text
530,Privacy Rule (2000),0.228488,According to privacy_rule_2000: e sanctions. These sanctions do not apply to...
852,Security Rule (2003),0.252265,According to security_rule_2003: §164.306(a). - (C) Sanction policy (Require...
818,NIST SP 800-66 Rev. 2 (2024),0.256860,According to nist_sp800-66r2: • Establish a frequency for reviewing policy a...



After reranking (top 3):


,category,rerank_score,text
1021,Privacy Rule (2000),0.890616,"According to privacy_rule_2000: g complaints, for at least six years from th..."
818,NIST SP 800-66 Rev. 2 (2024),0.857524,According to nist_sp800-66r2: • Establish a frequency for reviewing policy a...
530,Privacy Rule (2000),0.852086,According to privacy_rule_2000: e sanctions. These sanctions do not apply to...


### Measuring Search Accuracy (Bonus: Evaluation Metrics and Reranking)

**Business question:** Is the upgraded search *measurably* better, or does it just look better on a few examples?

**What we did:** Ran a controlled test on the **56 specific test questions** in the dataset. Each question was originally written from one particular passage. We asked each question and checked whether search returned *that exact passage*. We compared three setups:
- **MiniLM:** the course example's model (baseline)
- **Granite R2:** the upgraded embedding model
- **Granite R2 + rerank:** the two-stage search

**Metrics, in plain terms:**
- **Hit@1 / Hit@3 / Hit@5:** How often the correct passage appeared in the top 1, 3, or 5 results
- **MRR (Mean Reciprocal Rank):** Rewards putting the correct passage near the top. 1.0 is perfect, and 0 means it was never found

**Why it matters for HR:** This is the same discipline we would apply when evaluating any HR technology vendor: test on our own data and compare options side by side before buying in.

**How to read the results:**
- **MiniLM** found almost none of the correct passages. One likely reason is that MiniLM reads only about the first 256 tokens (~190 words) of each passage, so it never sees the rest of longer passages. We did not test this directly (for example, by re-running MiniLM on shortened passages), so treat it as a hypothesis, not a proven cause.
- **Granite R2** found the correct passage in the top 5 about 18% of the time.
- **Adding the reranker** raised that to about 32% and roughly **tripled MRR** (0.100 → 0.301).
- **These scores are a strict lower bound.** Only the *exact* source passage counts as a hit, but many questions have several valid answers from different regulations. The **relative** ranking of the three setups is the meaningful result.
- **Sample size matters.** With 56 questions, each question is worth about 1.8 percentage points, so a difference of a point or two would be noise. The gaps here are much larger than that, but they come from one dataset and one test set. They show the upgrade works better *for this library*, not that it is the best choice everywhere.
- Hit@3 and Hit@5 are identical after reranking because the reranker can only reorder the 20 passages it receives. If the right passage is not among them, reranking cannot recover it.

In [9]:
# Bonus: compare three retrieval setups on the same held-out test questions
# (up to 100 specific questions; generic "semantic_chunk" questions are excluded)
eval_pool = df[(df["qtype"] != "semantic_chunk") & (df["question"].str.len() > 0)]
eval_sample = eval_pool.sample(n=min(100, len(eval_pool)), random_state=RANDOM_SEED)
K_MAX = 5

# Baseline: Dr. Harun's original embedding model, with its own index
baseline_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
baseline_emb = baseline_model.encode(df["text"].tolist(), batch_size=64,
                                     convert_to_numpy=True, normalize_embeddings=True)
baseline_index = faiss.IndexFlatL2(baseline_emb.shape[1])
baseline_index.add(baseline_emb.astype("float32"))

def baseline_ids(q):
    e = baseline_model.encode([q], convert_to_numpy=True, normalize_embeddings=True)
    return list(baseline_index.search(e.astype("float32"), K_MAX)[1][0])

def granite_ids(q):
    return list(semantic_search(q, k=K_MAX).index)

def reranked_ids(q):
    return list(reranked_search(q, k=K_MAX, n_candidates=20).index)

def score(search_fn):
    ranks = []
    for doc_id, row in eval_sample.iterrows():
        hits = search_fn(row["question"])
        ranks.append(hits.index(doc_id) + 1 if doc_id in hits else None)
    return {
        "Hit@1": np.mean([r is not None and r <= 1 for r in ranks]),
        "Hit@3": np.mean([r is not None and r <= 3 for r in ranks]),
        "Hit@5": np.mean([r is not None and r <= 5 for r in ranks]),
        "MRR@5": np.mean([1 / r if r else 0 for r in ranks]),
    }

results = {
    "MiniLM (baseline)":   score(baseline_ids),
    "Granite R2":          score(granite_ids),
    "Granite R2 + rerank": score(reranked_ids),
}
comparison = pd.DataFrame(results).T
display(comparison.style.format({"Hit@1": "{:.1%}", "Hit@3": "{:.1%}", "Hit@5": "{:.1%}", "MRR@5": "{:.3f}"}))
print(f"Evaluated on {len(eval_sample)} held-out questions.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,Hit@1,Hit@3,Hit@5,MRR@5
MiniLM (baseline),0.0%,0.0%,1.8%,0.004
Granite R2,5.4%,14.3%,17.9%,0.100
Granite R2 + rerank,28.6%,32.1%,32.1%,0.301


Evaluated on 56 held-out questions.


## 7. Loading the Language Model

**Business question:** Which AI model should write the answers?

**What we did:** Loaded **IBM Granite 4.1 (3B, instruct version)**, a model trained to follow instructions, and ran a one-sentence test.

**Why this model instead of the course example (FLAN-T5):**
- **Room for evidence.** FLAN-T5 was trained to read about 512 tokens at a time, which is not enough to include several regulation passages in one prompt. Granite handles far longer inputs.
- **Follows structure.** In the course example, FLAN-T5 could not produce a structured executive summary. It repeated the instructions back instead.
- **Enterprise fit.** IBM designed Granite 4.1 for business uses, including retrieval-augmented answering, with governance, risk, and compliance reviews built into its development. It is Apache 2.0 licensed and runs on a free cloud GPU.

**How to read the results:** The test sentence should be a coherent, relevant answer. That confirms the model loaded correctly.

In [10]:
# Load the IBM Granite 4.1 3B instruct model (enterprise, compliance-focused, long context)
generation_model_name = "ibm-granite/granite-4.1-3b"

dtype = torch.float16 if device == "cuda" else torch.float32
tokenizer = AutoTokenizer.from_pretrained(generation_model_name)
model = AutoModelForCausalLM.from_pretrained(generation_model_name, dtype=dtype).to(device)
model.eval()

def text_generator(prompt: str, max_length: int = 256, do_sample: bool = False):
    """Send a prompt to the chat model and return [{"generated_text": ...}] (same format as the original lab)."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    generation_kwargs = {"max_new_tokens": max_length, "do_sample": do_sample,
                         "pad_token_id": tokenizer.eos_token_id}
    if do_sample:
        generation_kwargs["temperature"] = 0.7

    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_kwargs)

    # Keep only the newly generated text (not the prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    decoded_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return [{"generated_text": decoded_output}]

# Simple test prompt
test_output = text_generator("Explain in one sentence why hospitals train employees on patient privacy.", max_length=64)
print(test_output[0]["generated_text"])

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.15M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/6.10k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Hospitals train employees on patient privacy to protect sensitive health information and ensure compliance with legal and ethical standards, safeguarding patients' confidentiality and trust.


## 8. Retrieval-Augmented Generation (RAG)

**Business question:** Can the assistant answer HR questions using *only* the regulations it retrieved?

**What we did:** For each question, the assistant:
1. Retrieves the 5 most relevant passages and trims each to its first 1,200 characters, to keep the prompt a manageable size.
2. Places them in a prompt with the instruction: *answer using ONLY this context, and say so if the answer isn't there.*
3. Writes a concise, executive-ready answer, capped at 256 tokens (about 190 words).

**An HR analogy:** It's like handing an HR business partner five specific pages from the policy binder and asking them to answer from those pages only, not from memory.

**Questions tested:**
1. What training does HIPAA require for our workforce?
2. What steps must we take when a breach of unsecured patient information is discovered?
3. What are business associates responsible for under HIPAA?

**How to read the results:** The answers read like a knowledgeable compliance advisor wrote them. Sections 9 and 10 test whether every claim is actually supported by the retrieved passages.

In [11]:
def build_rag_prompt(question: str, retrieved_docs: List[str]) -> str:
    """
    Build a prompt that instructs the model to answer using only the provided context.
    """
    context_block = "\n\n".join([f"- {doc}" for doc in retrieved_docs])
    prompt = (
        "You are a HIPAA compliance advisor supporting the HR and Compliance teams "
        "at a healthcare organization. "
        "Answer the question using ONLY the information in the context below. "
        "If the context does not contain the answer, say that the information is not available.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {question}\n"
        "Answer in a concise, executive-ready paragraph."
    )
    return prompt

def rag_answer(question: str, k: int = 5, max_context_chars: int = 1200) -> Tuple[str, List[str]]:
    """
    Perform RAG:
    - Retrieve top-k documents
    - Truncate each document to a manageable size
    - Generate an answer
    """
    results = semantic_search(question, k=k)
    docs = results["text"].tolist()
    truncated_docs = [d[:max_context_chars] for d in docs]

    prompt = build_rag_prompt(question, truncated_docs)
    output = text_generator(prompt, max_length=256, do_sample=False)[0]["generated_text"]
    return output, truncated_docs

In [12]:
business_questions = [
    "What training does HIPAA require for our workforce?",
    "What steps must we take when a breach of unsecured patient information is discovered?",
    "What are business associates responsible for under HIPAA?",
]

for q in business_questions:
    print("\n" + "=" * 80)
    print("Question:", q)
    answer, _ = rag_answer(q)
    print("\nGenerated Answer:\n" + wrap(answer))


Question: What training does HIPAA require for our workforce?

Generated Answer:
HIPAA requires that workforce members receive security awareness and training as an
administrative safeguard (NIST SP 800-66r2, Sec. 5.5.2 and Sec. 5.1.5). This training
should be developed based on a Training Needs Assessment that identifies specific
security training requirements, monitors current threats (such as phishing and
ransomware), addresses any gaps in knowledge regarding organization devices or
technologies that pose risks to ePHI, and prioritizes content and audience. The training
must be documented in written form (electronic acceptable) and updated regularly to
ensure workforce members are adequately prepared to protect protected health information
(PHI).

Question: What steps must we take when a breach of unsecured patient information is discovered?

Generated Answer:
Upon discovering a breach of unsecured patient information, the healthcare organization
must promptly (within 60 calendar d

### Interpretation of the RAG Answers

| Question | What the answer got right | What needs verification | Business takeaway |
|---|---|---|---|
| Workforce training | Consistent with the NIST training-needs-assessment passages retrieved in Section 6 | Cites "NIST SP 800-66r2, Sec. 5.5.2 and Sec. 5.1.5." These section numbers cannot be confirmed from this notebook's output | A solid starting outline for a training program. Confirm the references before citing them in policy |
| Breach response | The 60-day deadline matches the Breach Notification Rule passages from Section 6, and the listed notice contents match the rule's requirements for individual notices | Oversimplifies HHS reporting (breaches of 500+ people: within 60 days; fewer: annual log) and labels it the Privacy Rule, when it is a separate rule | A useful checklist, but this timing error could lead to a late or unnecessary HHS filing |
| Business associates | Reflects the HITECH Omnibus Rule's shift to direct liability for business associates and the need for BAAs with subcontractors | Says business associates are directly liable for "any Security Rule violations committed by" subcontractors. That overstates it: they are directly liable for their *own* compliance, including obtaining BAAs from subcontractors | Adequate orientation for HR and benefits staff who manage vendor agreements, with the liability point confirmed by counsel |

**Overall:** All three answers were useful, and **all three contained at least one detail a reviewer would need to check.** This cell does not print the passages behind each answer, so a reviewer cannot trace those details from the output alone. Section 10 tests a prompt designed to close that gap.

## 9. Prompt Engineering Experiments

**Business question:** Does *how* we ask change how useful the answer is to an HR leader?

**What we did:** Asked the same two questions two ways:

| | Generic prompt | Structured executive prompt |
|---|---|---|
| Instructions | None, just "Answer:" | Answer as the Director of HR Compliance, using only the context |
| Required format | None | Summary, Key HIPAA Requirements, Recommended Actions for HR |
| Context provided | 2 passages | 5 passages |
| Maximum answer length | 256 tokens | 512 tokens |

**Why it matters for HR:** Leaders need answers they can use in a briefing or policy memo. Format affects whether the output is usable, and the amount of evidence affects whether it is complete.

**How to read the results:** Compare the two versions for each question on structure, relevance to HR, and whether every statement traces back to the retrieved passages.

In [13]:
def build_generic_prompt(question: str, retrieved_docs: List[str]) -> str:
    context_block = "\n\n".join(retrieved_docs)
    return (
        f"Context:\n{context_block}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

def build_structured_executive_prompt(question: str, retrieved_docs: List[str]) -> str:
    context_block = "\n\n".join([f"- {doc}" for doc in retrieved_docs])
    return (
        "You are the Director of HR Compliance at a healthcare organization. Using ONLY the context below, "
        "write an executive answer to the question. Your answer should include:\n"
        "- A brief summary\n"
        "- Key HIPAA requirements\n"
        "- One or two recommended actions for HR.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Executive Question: {question}\n"
        "Executive Answer:"
    )

def compare_prompts(question: str, max_context_chars: int = 1200):
    # Retrieve documents once
    results = semantic_search(question, k=5)
    docs = [d[:max_context_chars] for d in results["text"].tolist()]

    short_context = docs[:2]
    long_context = docs[:5]

    # Generic + short
    generic_short = text_generator(
        build_generic_prompt(question, short_context),
        max_length=256, do_sample=False
    )[0]["generated_text"]

    # Structured + long
    structured_long = text_generator(
        build_structured_executive_prompt(question, long_context),
        max_length=512, do_sample=False
    )[0]["generated_text"]

    print("\n" + "=" * 80)
    print("Question:", question)
    print("\n[Generic Prompt + Short Context]\n")
    print(wrap(generic_short))
    print("\n[Structured Executive Prompt + Expanded Context]\n")
    print(wrap(structured_long))

In [14]:
# Run prompt comparison on two HR compliance questions
compare_prompts("What administrative safeguards must we have in place to protect patient data?")
compare_prompts("How should we respond if an employee improperly accesses patient records?")


Question: What administrative safeguards must we have in place to protect patient data?

[Generic Prompt + Short Context]

According to the context provided, the administrative safeguards that a covered entity
(such as a healthcare provider or health plan) must have in place to protect patient
data include:

1. **Developing and maintaining policies and procedures**: The entity must establish
policies and procedures that address the security and privacy of electronic protected
health information (ePHI). These documents should outline the roles and responsibilities
of staff, the process for handling breaches, and the overall approach to maintaining
security.

2. **Training and awareness**: Staff must receive regular training on security and
privacy policies, as well as on the importance of protecting patient data. This training
should cover topics such as identifying and reporting security incidents, handling ePHI
appropriately, and understanding the regulations governing data security.

### Reflection on Prompt Quality: Comparing the Actual Outputs

| Question | Generic prompt + 2 passages (256-token cap) | Structured prompt + 5 passages (512-token cap) |
|---|---|---|
| Administrative safeguards | Four numbered items with no HR framing. Lists unique user IDs, a **technical** safeguard, as administrative | Clean Summary / Requirements / Actions format. Lists **three** technical safeguards (authentication, transmission security, integrity controls) as administrative. Adds "annual refresher courses" |
| Employee improperly accesses records | **Cut off mid-sentence** at step 6. Focuses on reviewing data-integrity processes rather than on the employee | Complete, with HR-specific steps: investigate, apply sanctions, retrain, strengthen monitoring. Adds "termination of employment if the breach was willful or severe" |

**Did the structured prompt make answers more usable?** Yes, in both cases. The Summary / Requirements / Actions format could go straight into an HR leadership briefing, and the "Director of HR Compliance" role kept the focus on HR actions.

**Did it reduce unsupported or incorrect content?** **No. It increased it.** The structured answers misclassified *more* safeguards (three vs. one) and added HR recommendations that are common practice but are not HIPAA requirements. HIPAA requires "appropriate" sanctions without specifying termination, and it does not set an annual training frequency. Longer, more confident answers gave the model more room to fill gaps with general knowledge.

**Caveat on the experiment:** Each comparison changed **three** things at once: prompt style, amount of context (2 vs. 5 passages), and maximum answer length (256 vs. 512 tokens). The generic answer's truncation is at least partly caused by its shorter limit, so the comparison is not fully fair. A cleaner test would change one factor at a time.

**Takeaway:** Prompt structure improves *usability*, not *accuracy*. Verification is still required.

## 10. Validation: Catching Unsupported Claims

**Business question:** Can we trust what the assistant says, and how would we know if we can't?

**What we did:** Asked a high-stakes HR question: *"What sanctions should we apply to workforce members who violate HIPAA policies?"* Then we printed the exact passages the answer was based on and compared the answer against them line by line.

**Why it matters for HR:** Discipline decisions carry legal risk. An answer that sounds authoritative but adds requirements the regulation doesn't contain could lead to inconsistent or indefensible discipline.

**How to read the results:** Any statement in the answer that doesn't appear in the retrieved passages below is **unsupported**, however reasonable it sounds.

In [15]:
# Choose a question and generate an answer + keep the context
question = "What sanctions should we apply to workforce members who violate HIPAA policies?"
answer, context_docs = rag_answer(question)

print("Question:", question)
print("\nGenerated Answer:\n" + wrap(answer))

Question: What sanctions should we apply to workforce members who violate HIPAA policies?

Generated Answer:
According to the HIPAA Security Rule (§164.306(a)(C)), covered entities are required to
establish and apply appropriate sanctions against workforce members who fail to comply
with the organization’s security policies and procedures. These sanctions may include
reprimands, warnings, suspension, or termination, depending on the severity of the
violation. However, the Privacy Rule (§164.530(e)(1)) clarifies that such sanctions do
not apply to workforce members who engage in whistleblower disclosures as defined in
§164.502(j) or to complaints, investigations, or opposition covered by §164.530(g)(2).
Therefore, when enforcing sanctions, HR and Compliance teams should ensure that any
disciplinary actions taken against workforce members for HIPAA policy violations do not
inadvertently penalize legitimate whistleblower activities or protected disclosures.


In [16]:
# Print the retrieved context so we can manually check whether the answer is supported
print("\n--- Retrieved Context Snippets (for manual verification) ---\n")
for i, doc in enumerate(context_docs):
    print(f"[Doc {i+1}]\n{wrap(doc[:400])}...\n")


--- Retrieved Context Snippets (for manual verification) ---

[Doc 1]
According to nist_sp800-66r2: • Establish a frequency for reviewing policy and
procedures. | • Is there a process for communicating policies and procedures to the
affected employees? • Are policies and procedures reviewed and updated as needed? | | 6.
Develop and Implement a Sanction Policy 42 Implementation Specification (Required) | •
Apply appropriate sanctions against workforce members who fai...

[Doc 2]
According to privacy_rule_2000: §164.530(e)(1) but modify the proposed provision on
sanctions to clarify that the sanctions required under this rule do not apply to
workforce members of a covered entity for whistleblower disclosures. ## Disclosures by
Workforce Members Who Are Crime Victims The proposed rule did not address disclosures by
workforce members who are victims of a crime. In the final...

[Doc 3]
According to hitech_omnibus_2013: ituation will determine whether reputational harm has
occurred, such a

### Unsupported Claims Found (Real Examples from This Run)

**Claim 1: invented content.**
> *"These sanctions may include reprimands, warnings, suspension, or termination, depending on the severity of the violation."*

**None of the five retrieved passages lists types of sanctions.** The regulation requires *"appropriate sanctions"* and leaves the specifics to each organization. The model filled the gap with general HR knowledge. The list is plausible, but an employee or manager could easily mistake it for a federal requirement.

**Claim 2: wrong citation.**
The answer cites **§164.306(a)(C)**. The Security Rule's sanction policy standard is **§164.308(a)(1)(ii)(C)**. The model appears to have stitched together a fragment from the start of Document 5 (*"§164.306(a). - (C) Sanction policy"*), where one section ends and the next begins. A wrong citation in a compliance document undermines credibility with auditors and counsel.

**Other errors observed in this notebook:**
- The breach answer states that HHS must be notified "within the same timeframe" for all breaches. Only breaches affecting **500 or more** people are reported to HHS within 60 days. Smaller breaches are logged and reported annually.
- The same answer files breach notification under the Privacy Rule. It is a separate rule (the Breach Notification Rule).

**Mitigation tested below:** Require the model to quote supporting text and label each quote with its document number and source.

In [17]:
def rag_answer_with_citations(question: str, k: int = 5, max_context_chars: int = 1200) -> Tuple[str, List[str]]:
    results = semantic_search(question, k=k)
    docs = results["text"].tolist()
    sources = results["category"].tolist()
    truncated_docs = [d[:max_context_chars] for d in docs]

    # Label each document with its number AND its source regulation for traceability
    context_block = "\n\n".join(
        [f"[Doc {i+1} - {src}] {doc}" for i, (src, doc) in enumerate(zip(sources, truncated_docs))]
    )
    prompt = (
        "You are a careful HIPAA compliance analyst. Using ONLY the information in the context below, "
        "answer the question. After your answer, list 1-3 short quotes from the context "
        "that directly support your answer, with the Doc number. If the context does not "
        "support an answer, say that clearly.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {question}\n"
        "Answer with citations:"
    )

    output = text_generator(prompt, max_length=256, do_sample=False)[0]["generated_text"]
    return output, truncated_docs

validated_answer, _ = rag_answer_with_citations(question)
print("\nValidated Answer with Citations:\n")
print(wrap(validated_answer))


Validated Answer with Citations:

According to NIST SP 800-66 Rev. 2 (2024), Section 6. Develop and Implement a Sanction
Policy, it states:

• "Apply appropriate sanctions against workforce members who fail to comply with the
security policies and procedures of the covered entity or business associate." [Doc 1]

Additionally, the HIPAA Privacy Rule (2000) §164.530(e)(1) (with modifications) also
addresses sanctions, noting:

• "These sanctions do not apply to whistleblower disclosures." [Doc 2]

Furthermore, the HIPAA Security Rule (2003) §164.306(a)(C) requires:

• "Apply appropriate sanctions against workforce members who fail to comply with the
security policies and procedures of the covered entity." [Doc 5]

These citations collectively outline the required sanctions for workforce members who
violate HIPAA policies, with the exception of whistleblower disclosures as specified.


### Automated Check: Are the Quotes and Citations Real?

**Business question:** Can we check the citation-prompt answer automatically, instead of relying only on a human reading it line by line?

**What we did:** Wrote a check that runs on the model's output with no additional AI:
1. **Quote check.** Finds every quoted passage in the citation-prompt answer and searches the five retrieved passages for it. A quote is *Verbatim* if it appears word-for-word, *Partial match* if at least half of it appears as one continuous exact span (the model edited or paraphrased it), and *Not found* otherwise.
2. **Citation check.** Finds every CFR section cited in *both* answers and checks whether that exact citation appears in the retrieved text.

**Why it matters for HR:** A reviewer's time is limited. An automated first pass flags exactly which quotes and citations need a human to look them up, instead of every line of every answer.

**How to read the results:** *Verbatim* and *Yes* mean the text can be traced to a source. Anything else needs human review. **Limitation:** a citation that appears in the retrieved text can still be the wrong citation for the claim, so this check narrows review but does not replace it.

In [18]:
# Automated check: are the model's quotes really in the sources, and do its CFR citations appear there?
import re
from difflib import SequenceMatcher

# Rebuild the exact context the citation prompt saw (same search, same 1,200-character trim)
citation_docs = [d[:1200] for d in semantic_search(question, k=5)["text"].tolist()]

def normalize(s: str) -> str:
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return re.sub(r"\s+", " ", s).strip().lower()

def check_quote(quote: str, docs: List[str]):
    """Return (verdict, closest doc number, share of the quote found as one continuous exact span)."""
    q = normalize(quote).rstrip(".")
    best_share, best_doc = 0.0, None
    for i, d in enumerate(docs, start=1):
        dn = normalize(d)
        if q in dn:
            return "Verbatim", i, 1.0
        m = SequenceMatcher(None, q, dn, autojunk=False).find_longest_match(0, len(q), 0, len(dn))
        share = m.size / max(len(q), 1)
        if share > best_share:
            best_share, best_doc = share, i
    verdict = "Partial match (edited or paraphrased)" if best_share >= 0.5 else "Not found in sources"
    return verdict, best_doc, best_share

# 1. Check every quoted passage in the citation-prompt answer
quotes = re.findall(r'["“]([^"”]{15,})["”]', validated_answer)
quote_rows = []
for qt in quotes:
    verdict, doc_no, share = check_quote(qt, citation_docs)
    quote_rows.append({
        "Quote from answer": qt[:90] + ("..." if len(qt) > 90 else ""),
        "Result": verdict,
        "Closest source": f"Doc {doc_no}" if doc_no else "-",
        "Longest exact overlap": f"{share:.0%}",
    })
print("Quote verification (citation-prompt answer):")
display(pd.DataFrame(quote_rows))

# 2. Check every CFR citation in BOTH answers against the retrieved text
cfr_pattern = r"§\s?\d{3}\.\d+(?:\s?\([A-Za-z0-9]+\))*"
all_context = normalize(" ".join(citation_docs)).replace(" ", "")
cite_rows = []
for cite in sorted(set(re.findall(cfr_pattern, answer + " " + validated_answer))):
    compact = cite.replace(" ", "")
    found = normalize(compact).replace(" ", "") in all_context
    cite_rows.append({
        "CFR citation in answers": compact,
        "Appears exactly in retrieved text?": "Yes" if found else "No - verify on eCFR",
    })
print("\nCitation verification (both answers):")
display(pd.DataFrame(cite_rows))


n_verbatim = sum(r["Result"] == "Verbatim" for r in quote_rows)
n_cited = sum(r["Appears exactly in retrieved text?"] == "Yes" for r in cite_rows)
print(f"\nSummary: {n_verbatim} of {len(quote_rows)} quotes are verbatim; "
      f"{n_cited} of {len(cite_rows)} CFR citations appear exactly in the retrieved text.")

Quote verification (citation-prompt answer):


,Quote from answer,Result,Closest source,Longest exact overlap
0,Apply appropriate sanctions against workforce members who fail to comply wit...,Verbatim,Doc 1,100%
1,These sanctions do not apply to whistleblower disclosures.,Partial match (edited or paraphrased),Doc 4,81%
2,Apply appropriate sanctions against workforce members who fail to comply wit...,Verbatim,Doc 1,100%



Citation verification (both answers):


,CFR citation in answers,Appears exactly in retrieved text?
0,§164.306(a)(C),No - verify on eCFR
1,§164.502(j),Yes
2,§164.530(e)(1),Yes
3,§164.530(g)(2),Yes



Summary: 2 of 3 quotes are verbatim; 3 of 4 CFR citations appear exactly in the retrieved text.


### Interpretation: Did Citations Fix the Problem?

| Check | Standard RAG answer | Citation-prompt answer |
|---|---|---|
| Invented list of discipline types | **Present** (reprimands, warnings, suspension, termination) | **Removed** |
| Wrong citation §164.306(a)(C) | **Present** | **Still present** (flagged by the automated check) |
| Whistleblower exception | Stated correctly | Stated correctly, but the "quote" is paraphrased |
| Quotes a reviewer can trace to a source | None given | Only those marked *Verbatim* by the automated check (see its summary line); the rest are flagged for human review |

**What improved:** The *most serious* error, invented content presented as federal guidance, disappeared. Every remaining statement now points to a numbered source, so a reviewer knows where to look.

**What didn't:** The wrong citation persisted, and the model presented edited text as direct quotes. The problems shifted from *invented facts* to *inaccurate attribution*. That is less dangerous, but still unacceptable in a compliance document, because it misleads anyone who relies on the quote marks.

**Takeaway for HR:** Asking for citations *reduces* the severity of unsupported claims but does not eliminate them. The automated check above catches some attribution errors instantly, but not all. For any answer used in policy, training, or discipline decisions:
1. Run the automated quote and citation check.
2. Have a Privacy or Compliance Officer review anything it flags, plus a sample of what it passes.
3. Confirm every CFR citation against the official eCFR text before the answer is used.

## 11. Managerial Summary

**What types of questions does this system answer well?**
The assistant performs best on well-defined compliance questions tied to a specific regulation: breach notification deadlines, whether a sanction policy is required, what training the Security Rule expects. For these, search consistently surfaced the right federal source, and the language model produced clear, well-organized answers that an HR leader could use as a starting point for a briefing or policy review. It also saves real time. Finding the same passages manually across thousands of pages of preambles and guidance would take an HR generalist hours.

**What are the limitations and risks in a real consulting environment?**
The biggest risk is **confident error**. The model invented discipline examples the regulation does not list, cited the wrong CFR section, and misclassified technical safeguards as administrative. Because the writing is fluent and professional, these errors are harder to spot than obvious mistakes. In HR, that can translate into inconsistent discipline, inaccurate training content, or findings in an audit. There are data limits as well. The library is weighted heavily toward the Privacy Rule, the core regulation text failed to extract, and it contains no state law, no HHS enforcement guidance, and no organization-specific policies. Finally, search accuracy on exact passages was modest even after improvements, so some questions will be answered from related rather than ideal sources.

**How would you improve this system for enterprise deployment?**
1. **Complete the library** with the official eCFR regulation text, HHS guidance, applicable state privacy laws, and the organization's own HIPAA policies and sanction policy.
2. **Keep reranking** in the pipeline, and track accuracy metrics over time as the library grows.
3. **Expand the automated quote and citation check** prototyped in Section 10 so every answer is screened before it is shown.
4. **Require human review** by the Privacy or Compliance Officer for any answer used in policy, training, or discipline decisions, and label all outputs "not legal advice."
5. **Govern access and usage.** Keep patient data out of prompts, log questions and answers for audit, and review a sample of answers each quarter.

Used as a research aid with human oversight, this assistant could meaningfully reduce the time HR spends on routine compliance questions while keeping accountability with qualified staff.

### Declaration of Generative AI Usage

During the preparation of the code modifications, model selection, and markdown annotations in this notebook, **Claude (Claude Opus 5.5), Anthropic** was used to adapt the instructor's example code to a new dataset, research model options, debug errors, and draft explanatory text. After using this tool, I reviewed, ran, and edited all content as needed and take full responsibility for the content of the submission.

The course example notebook was originally created by the instructor with the support of Google Colab's Gemini model.

---
### Appendix: Exporting This Notebook
The cell below captures the notebook exactly as run, downloads it as .ipynb and .html, and the .html is then printed to PDF from the browser.

In [ ]:
# Export: grabs the notebook exactly as it appears on screen, with every output,
# so it never picks up an older or half-saved copy from Drive.
# Downloads both files you submit: the .ipynb and the .html (print the .html to PDF).
import json
from google.colab import _message, files

EXPORT_NAME = "HW_SemanticSearch_Generation_JenkinsVeronica"
nb_json = _message.blocking_request("get_ipynb", request="", timeout_sec=120)["ipynb"]
with open(f"/content/{EXPORT_NAME}.ipynb", "w") as f:
    json.dump(nb_json, f)

!jupyter nbconvert --to html "/content/{EXPORT_NAME}.ipynb"
files.download(f"/content/{EXPORT_NAME}.ipynb")
files.download(f"/content/{EXPORT_NAME}.html")